In [ ]:
#@title MASTER — Persistent OpenVL + FrankaPick + SimplerEnv Setup
!apt-get update -qq
!apt-get install -y -qq python3.12-venv

# ============================================================
# Persistent OpenVLA environment on Google Drive
# Installs only when required versions are missing/different
# ============================================================

import os
import sys
import subprocess
import shutil
import importlib.metadata as metadata

from google.colab import drive

# ------------------------------------------------------------
# 1. Mount Google Drive
# and Persistent Hugging Face cache
# ------------------------------------------------------------

drive.mount("/content/drive")


HF_CACHE = "/content/drive/MyDrive/huggingface_cache"

os.makedirs(HF_CACHE, exist_ok=True)

os.environ["HUGGINGFACE_HUB_CACHE"] = HF_CACHE

print("Hugging Face cache:", HF_CACHE)

ENV_PATH = "/content/drive/MyDrive/colab_env_openvla"
VENV_PYTHON = os.path.join(ENV_PATH, "bin", "python")

'''
if os.path.exists(ENV_PATH):
    print("Removing old/non-venv directory:")
    print(ENV_PATH)
    shutil.rmtree(ENV_PATH)
'''

print("Creating fresh persistent venv...")

# ------------------------------------------------------------
# 2. Versions we require
# ------------------------------------------------------------

REQUIRED = {
    "transformers": "4.40.1",
    "tokenizers": "0.19.1",
    "accelerate": "0.26.0",
    "timm": "0.9.10",
    "peft": "0.11.1",
    "sentencepiece": "0.1.99",
    "draccus": "0.8.0",
}

# ------------------------------------------------------------
# 3. Create persistent venv if necessary
# ------------------------------------------------------------

if not os.path.exists(VENV_PYTHON):

    print("================================================")
    print("Creating persistent OpenVLA environment")
    print("================================================")

    subprocess.run(
        [
            sys.executable,
            "-m",
            "venv",
            "--system-site-packages",
            ENV_PATH,
        ],
        check=True,
    )

    print("✅ Virtual environment created")
else:
    print("✅ Persistent environment already exists")


# ------------------------------------------------------------
# 4. Check installed versions INSIDE the venv
# ------------------------------------------------------------

def get_installed_version(package):
    """
    Ask the persistent venv's Python which version is installed.
    """
    result = subprocess.run(
        [
            VENV_PYTHON,
            "-c",
            (
                "import importlib.metadata as m; "
                f"print(m.version('{package}'))"
            ),
        ],
        capture_output=True,
        text=True,
    )

    if result.returncode != 0:
        return None

    return result.stdout.strip()


print("\n================================================")
print("Checking OpenVLA environment")
print("================================================")

needs_install = []

for package, required_version in REQUIRED.items():

    installed = get_installed_version(package)

    if installed is None:
        print(f"❌ {package}: NOT INSTALLED")
        needs_install.append(package)

    elif required_version is None:
        print(f"✅ {package}: {installed}")

    elif installed != required_version:
        print(
            f"⚠️ {package}: {installed} "
            f"(required {required_version})"
        )
        needs_install.append(package)

    else:
        print(f"✅ {package}: {installed}")


# ------------------------------------------------------------
# 5. Install ONLY if something is missing/wrong
# ------------------------------------------------------------

if needs_install:

    print("\n================================================")
    print("Installing/updating OpenVLA dependencies")
    print("================================================")

    packages = [
        "transformers==4.40.1",
        "tokenizers==0.19.1",
        "accelerate==0.26.0",
        "timm==0.9.10",
        "peft==0.11.1",
        "sentencepiece==0.1.99",
        "draccus==0.8.0",
    ]

    subprocess.run(
        [
            VENV_PYTHON,
            "-m",
            "pip",
            "install",
            "-q",
            *packages,
        ],
        check=True,
    )

    print("✅ Installation/update complete")

else:

    print("\n================================================")
    print("🎉 OpenVLA environment already satisfies all")
    print("   required versions.")
    print("   SKIPPING pip installation.")
    print("================================================")


# ------------------------------------------------------------
# 6. Make the venv packages available to THIS notebook
# ------------------------------------------------------------

# We are using Colab's Python kernel, so add the venv's
# site-packages to sys.path.

PYTHON_VERSION = f"{sys.version_info.major}.{sys.version_info.minor}"

VENV_SITE = os.path.join(
    ENV_PATH,
    "lib",
    f"python{PYTHON_VERSION}",
    "site-packages",
)

if os.path.exists(VENV_SITE):

    if VENV_SITE not in sys.path:
        sys.path.insert(0, VENV_SITE)

    print("\nVenv site-packages:")
    print(VENV_SITE)

else:

    print("\n⚠️ WARNING:")
    print("Could not find:")
    print(VENV_SITE)


# ------------------------------------------------------------
# 7. Final verification
# ------------------------------------------------------------

print("\n================================================")
print("FINAL ENVIRONMENT CHECK")
print("================================================")

print("Notebook Python:")
print(sys.version)

print("\nNotebook executable:")
print(sys.executable)


#import transformers
#import tokenizers
#import accelerate
#import timm
#import peft

#print("\nTransformers :", transformers.__version__)
#print("Tokenizers   :", tokenizers.__version__)
#print("Accelerate   :", accelerate.__version__)
#print("Timm         :", timm.__version__)
##print("PEFT         :", peft.__version__)

#print("\nTransformers loaded from:")
#print(transformers.__file__)

print("\n✅ Persistent OpenVLA environment ready.")

!pip install -q mujoco==3.8.0 mediapy

print("mediapy and mujoco installed")

In [ ]:
# ============================================================
# AUTOMATIC CAMERA SEARCH FOR PANDA / MuJoCo
#
# CPU + OSMesa
#
# Generates contact sheets of candidate camera views.
# ============================================================

# ------------------------------------------------------------
# 1. Environment
# ------------------------------------------------------------

import os

os.environ["MUJOCO_GL"] = "osmesa"
os.environ.pop("PYOPENGL_PLATFORM", None)

# ------------------------------------------------------------
# 2. Imports
# ------------------------------------------------------------

import numpy as np
import mujoco
import mediapy as media
from PIL import Image, ImageDraw, ImageFont

print("MuJoCo:", mujoco.__version__)
print("MUJOCO_GL:", os.environ.get("MUJOCO_GL"))


# ------------------------------------------------------------
# 3. Load Panda model
# ------------------------------------------------------------

model_path = (
    "/content/drive/MyDrive/"
    "franka_emika_panda/"
    "vla_task.xml"
)

model = mujoco.MjModel.from_xml_path(model_path)
data = mujoco.MjData(model)

print("MODEL LOADED")


# ------------------------------------------------------------
# 4. Ready pose
# ------------------------------------------------------------

ready_qpos = [
    0.0,
    -0.785,
    0.0,
    -2.356,
    0.0,
    1.571,
    0.785
]

data.qpos[:7] = ready_qpos

mujoco.mj_forward(model, data)

print("READY POSE SET")


# ------------------------------------------------------------
# 5. Find pinch site
# ------------------------------------------------------------

site_id = mujoco.mj_name2id(
    model,
    mujoco.mjtObj.mjOBJ_SITE,
    "pinch"
)

if site_id < 0:
    raise RuntimeError("Could not find pinch site")

pinch = data.site_xpos[site_id].copy()

print("Panda pinch:", np.round(pinch, 4))


# ------------------------------------------------------------
# 6. Find camera
# ------------------------------------------------------------

cam_id = mujoco.mj_name2id(
    model,
    mujoco.mjtObj.mjOBJ_CAMERA,
    "vla_camera"
)

if cam_id < 0:
    raise RuntimeError("Could not find vla_camera")

print("Camera ID:", cam_id)


# ============================================================
# CAMERA SEARCH PARAMETERS
# ============================================================

# Camera azimuth in degrees.
#
# 0 degrees   = +X
# 90 degrees  = +Y
# 180 degrees = -X
# 270 degrees = -Y
#
# We will search all four quadrants.

AZIMUTHS = [
    0,
    45,
    90,
    135,
    180,
    225,
    270,
    315,
]

# Elevation above the XY plane.

ELEVATIONS = [
    20,
    30,
    40,
    50,
]

# Distance from the target.

DISTANCES = [
    0.8,
    1.0,
    1.2,
]

# Camera vertical field of view.

FOVS = [
    40,
    50,
    60,
]


# ------------------------------------------------------------
# To keep the contact sheets manageable, initially use
# ONE FOV and ONE distance.
#
# We can expand the search later.
# ------------------------------------------------------------

SEARCH_DISTANCE = 1.0
SEARCH_FOV = 50.0


# ============================================================
# CAMERA TARGET
# ============================================================

# We do NOT aim directly at the pinch point.
#
# Instead, target a point between the Panda base and the
# end effector, slightly above the middle of the workspace.
#
# Adjust these numbers if necessary.

BASE = np.array([0.0, 0.0, 0.0])

target = (
    0.55 * pinch +
    0.45 * BASE
)

# Raise target slightly so the image contains more of
# the arm/workspace rather than putting the hand exactly
# at the optical center.

target[2] += 0.08

print("\nCamera target:")
print(np.round(target, 4))


# ============================================================
# LOOK-AT FUNCTION
# ============================================================

def look_at(camera_pos, target):

    forward = target - camera_pos
    forward /= np.linalg.norm(forward)

    world_up = np.array([
        0.0,
        0.0,
        1.0
    ])

    right = np.cross(
        forward,
        world_up
    )

    right /= np.linalg.norm(right)

    up = np.cross(
        right,
        forward
    )

    up /= np.linalg.norm(up)

    # MuJoCo camera:
    #
    # +X = right
    # +Y = up
    # -Z = viewing direction

    R = np.column_stack([
        right,
        up,
        -forward
    ])

    return R


# ============================================================
# SET CAMERA
# ============================================================

def set_camera(
    azimuth_deg,
    elevation_deg,
    distance,
    fov,
):

    az = np.radians(azimuth_deg)
    el = np.radians(elevation_deg)

    # Spherical coordinates around target.
    #
    # x/y determine azimuth
    # z determines elevation

    camera_pos = target + np.array([
        distance * np.cos(el) * np.cos(az),
        distance * np.cos(el) * np.sin(az),
        distance * np.sin(el),
    ])

    cam_R = look_at(
        camera_pos,
        target
    )

    cam_quat = np.zeros(4)

    mujoco.mju_mat2Quat(
        cam_quat,
        cam_R.flatten()
    )

    model.cam_pos[cam_id] = camera_pos
    model.cam_quat[cam_id] = cam_quat
    model.cam_fovy[cam_id] = fov

    mujoco.mj_forward(model, data)

    return camera_pos, cam_quat


# ============================================================
# RENDERER
# ============================================================

renderer = mujoco.Renderer(
    model,
    height=224,
    width=224
)

print("RENDERER CREATED")


# ============================================================
# RENDER ONE CAMERA
# ============================================================

def render_camera(
    azimuth,
    elevation,
    distance,
    fov
):

    camera_pos, cam_quat = set_camera(
        azimuth,
        elevation,
        distance,
        fov
    )

    renderer.update_scene(
        data,
        camera=cam_id
    )

    pixels = renderer.render()

    return pixels, camera_pos, cam_quat


# ============================================================
# CREATE CONTACT SHEET
# ============================================================

def make_contact_sheet(
    images,
    labels,
    columns=4,
    tile_size=224
):

    rows = int(
        np.ceil(len(images) / columns)
    )

    label_height = 30

    sheet = Image.new(
        "RGB",
        (
            columns * tile_size,
            rows * (tile_size + label_height)
        ),
        "white"
    )

    draw = ImageDraw.Draw(sheet)

    for i, (pixels, label) in enumerate(
        zip(images, labels)
    ):

        row = i // columns
        col = i % columns

        x = col * tile_size
        y = row * (tile_size + label_height)

        img = Image.fromarray(pixels)

        sheet.paste(
            img,
            (x, y)
        )

        draw.text(
            (x + 5, y + tile_size + 5),
            label,
            fill="black"
        )

    return sheet


# ============================================================
# PHASE 1:
# SEARCH AZIMUTH + ELEVATION
# ============================================================

print("\n")
print("=" * 70)
print("PHASE 1: AZIMUTH + ELEVATION SEARCH")
print("=" * 70)

images = []
labels = []
camera_parameters = []

for elevation in ELEVATIONS:

    for azimuth in AZIMUTHS:

        pixels, camera_pos, cam_quat = render_camera(
            azimuth,
            elevation,
            SEARCH_DISTANCE,
            SEARCH_FOV
        )

        images.append(pixels)

        labels.append(
            f"A={azimuth}  E={elevation}"
        )

        camera_parameters.append({
            "azimuth": azimuth,
            "elevation": elevation,
            "distance": SEARCH_DISTANCE,
            "fov": SEARCH_FOV,
            "position": camera_pos.copy(),
            "quaternion": cam_quat.copy(),
        })


# ------------------------------------------------------------
# Display phase 1
# ------------------------------------------------------------

sheet = make_contact_sheet(
    images,
    labels,
    columns=4
)

print(
    f"Generated {len(images)} camera views"
)

media.show_image(
    np.asarray(sheet)
)


# ============================================================
# PHASE 2:
# DISTANCE SEARCH
#
# This uses a few promising-looking orientations,
# but displays all distances.
# ============================================================

# After looking at Phase 1, you can modify this list.
#
# Initially we'll use several common 3/4 views.

GOOD_AZIMUTHS = [
    135,
    180,
    225,
    270,
    315,
]

GOOD_ELEVATION = 35


print("\n")
print("=" * 70)
print("PHASE 2: DISTANCE SEARCH")
print("=" * 70)

images = []
labels = []

for azimuth in GOOD_AZIMUTHS:

    for distance in DISTANCES:

        pixels, camera_pos, cam_quat = render_camera(
            azimuth,
            GOOD_ELEVATION,
            distance,
            SEARCH_FOV
        )

        images.append(pixels)

        labels.append(
            f"A={azimuth}  D={distance}"
        )


sheet = make_contact_sheet(
    images,
    labels,
    columns=3
)

media.show_image(
    np.asarray(sheet)
)


# ============================================================
# PHASE 3:
# FOV SEARCH
# ============================================================

# Pick one promising orientation here after Phase 1.
#
# These are only defaults.

FINAL_AZIMUTH = 225
FINAL_ELEVATION = 35
FINAL_DISTANCE = 1.0


print("\n")
print("=" * 70)
print("PHASE 3: FOV SEARCH")
print("=" * 70)

images = []
labels = []

for fov in FOVS:

    pixels, camera_pos, cam_quat = render_camera(
        FINAL_AZIMUTH,
        FINAL_ELEVATION,
        FINAL_DISTANCE,
        fov
    )

    images.append(pixels)

    labels.append(
        f"FOV={fov}"
    )


sheet = make_contact_sheet(
    images,
    labels,
    columns=3
)

media.show_image(
    np.asarray(sheet)
)


# ============================================================
# PRINT FINAL CAMERA PARAMETERS
# ============================================================

camera_pos, cam_quat = set_camera(
    FINAL_AZIMUTH,
    FINAL_ELEVATION,
    FINAL_DISTANCE,
    SEARCH_FOV
)

print("\n")
print("=" * 70)
print("CURRENT CAMERA PARAMETERS")
print("=" * 70)

print("Azimuth :", FINAL_AZIMUTH)
print("Elevation:", FINAL_ELEVATION)
print("Distance:", FINAL_DISTANCE)
print("FOV:", SEARCH_FOV)

print("\nCamera position:")
print(
    np.round(camera_pos, 8)
)

print("\nCamera quaternion:")
print(
    np.round(cam_quat, 8)
)

print("\nTarget:")
print(
    np.round(target, 8)
)

print("\n")
print("=" * 70)
print("CAMERA SEARCH COMPLETE")
print("=" * 70)